# FranceCrops workflow outline

Markdown-only workflow outline for the Benguet thesis. No implementation code is included.

**Target:** cabbage, carrot, potato; Random Forest; Sentinel-2 L2A; four calendar months; eight semi-monthly composites; five bands plus NDVI, NDRE and NDMI.

Use each section to write your own implementation. Record decisions and findings under the prompts. This outline does not indicate that any stage has already been completed.


## 1. Define the study and data inventory

**Work to implement**

Specify the intended mapping outcome and common preprocessing/detection rules with Sen4AgriNet and Benguet. Record the FranceCrops release and upstream RPG year. The benchmark supplies cabbage and table potato; carrot is absent from its 20-class list.

**Expected output**

Study scope and source-combination plan.

**Your decisions and findings**

- Method and parameters:
- Checks performed:
- Results:
- Unresolved issues / next action:


## 2. Download benchmark and helper files

**Work to implement**

Inventory benchmark Parquet files and separate metadata, class_map and geolocation helpers. Start with small helper files to verify class counts and spatial availability. Preserve benchmark split membership; inventory does not mean merging all splits for training.

**Expected output**

File manifest, schemas and source versions.

**Your decisions and findings**

- Method and parameters:
- Checks performed:
- Results:
- Unresolved issues / next action:


## 3. Inspect the benchmark arrays and labels

**Work to implement**

Confirm x, y and parcel_id. Published shape: 100 sampled pixels × 60 dates × 12 bands. Confirm class-map IDs (cabbage 4, table potato 12) in the actual release. Values are L2A scaled by 10000; dates are five-day aligned from February 1 to November 23, 2022. Missing observations were interpolated, so these are not 60 guaranteed clear acquisitions. Count parcel rows and check duplicates.

**Expected output**

Verified schema, class distribution, selected parcel IDs and exploratory timelines.

**Your decisions and findings**

- Method and parameters:
- Checks performed:
- Results:
- Unresolved issues / next action:


## 4. Join labels to geolocation and decode boundaries

**Work to implement**

Join by verified parcel_id, not row position. Read geolocation documentation to confirm CRS. Decode WKB bytes where needed; check polygon validity and coordinate bounds. Verify cabbage category meaning against the intended target. The sampled pixels are not an image grid; geolocation boundaries do not restore their exact individual locations. Deduplicate fields and retain label/split provenance.

**Expected output**

Labeled field polygons with confirmed CRS and stable IDs.

**Your decisions and findings**

- Method and parameters:
- Checks performed:
- Results:
- Unresolved issues / next action:


## 5. Choose original-image retrieval for the final pipeline

**Work to implement**

Use supplied interpolated timelines for exploratory plots only unless a separate experiment explicitly evaluates that processing. Retrieve original L2A for the field boundaries to apply the same masks and actual-observation composites as other sources. Treat annual RPG labels as crop declarations, not planting/harvest records.

**Expected output**

Documented decision to use original acquisitions; exploratory and final inputs clearly identified.

**Your decisions and findings**

- Method and parameters:
- Checks performed:
- Results:
- Unresolved issues / next action:


## 6. Retrieve Sentinel-2 L2A

**Work to implement**

Query COPERNICUS/S2_SR_HARMONIZED for candidate field footprints. Retrieve a full-year timeline and adjacent months when needed, using the registered project ths-crop-map-project. Retain actual dates, image IDs, quality information, five target bands, and B2 for visual checks. Distinguish image granules from unique dates; establish how overlapping granules are mosaicked.

**Expected output**

Dated imagery inventory, field footprints, and source provenance.

**Your decisions and findings**

- Method and parameters:
- Checks performed:
- Results:
- Unresolved issues / next action:


## 7. Mask clouds and align inputs

**Work to implement**

Mask clouds, shadows, snow and invalid observations; retain valid bare soil. Review SCL screening and any additional cloud-quality method. Apply reflectance scaling consistently. Fix CRS, grid origin, resolution, extent and resampling. Use nearest-neighbor for labels. B5/B11 are 20 m: resampling does not add detail. Inspect overlays and interior-pixel buffers.

**Expected output**

Comparable cloud-masked imagery and documented preprocessing settings.

**Your decisions and findings**

- Method and parameters:
- Checks performed:
- Results:
- Unresolved issues / next action:


## 8. Create field-level growth timelines

**Work to implement**

Compute NDVI=(B8-B4)/(B8+B4), NDRE=(B8-B5)/(B8+B5), and NDMI=(B8-B11)/(B8+B11). Handle invalid denominators. Summarize field-interior pixels per acquisition; record clear fractions and supporting pixel counts. Plot dated RGB and indices in chronological order.

**Expected output**

Quality-controlled parcel timelines and visual review notes.

**Your decisions and findings**

- Method and parameters:
- Checks performed:
- Results:
- Unresolved issues / next action:


## 9. Develop a green-up detector

**Work to implement**

Identify a preceding low period and sustained growth for each candidate cycle. Choose a robust smoother or suitable single-cycle fit. Define robust cycle-specific baseline and peak estimators. Test relative-amplitude thresholds such as 15%, 20%, and 25% as experimental settings, not validated crop constants. Require observed support and subsequent clear observations within a defined time horizon. Use NDRE/RGB as supporting checks.

**Expected output**

Candidate onset date, observed uncertainty interval, fit diagnostics and rejection reasons.

**Your decisions and findings**

- Method and parameters:
- Checks performed:
- Results:
- Unresolved issues / next action:


## 10. Validate growth estimates and label relevance

**Work to implement**

Compare estimates with independent emergence/green-up evidence where available. Planting is not green-up; decline is not necessarily harvest. Annual crop labels must be linked to the selected cycle. Assess gaps, weak signals, multiple cycles, threshold sensitivity and region-specific failures. Calibrate on development fields and freeze rules before final testing. Without independent dates, report estimates or review agreement rather than verified planting accuracy.

**Expected output**

Frozen detector settings, uncertainty records and accepted field-season labels.

**Your decisions and findings**

- Method and parameters:
- Checks performed:
- Results:
- Unresolved issues / next action:


## 11. Select four consecutive calendar months

**Work to implement**

Proposed rule: start on the first day of the month containing accepted green-up, then take that month and the next three. Flag onset intervals crossing month boundaries. Check that the window corresponds to the declared crop and does not contain an unresolved rotation. Do not compress seasons or concatenate separated months. The same detector must work on unknown Benguet crops; field-level detection requires boundaries. Peak-based detection is retrospective and may need imagery beyond the classifier window.

**Expected output**

Accepted window start/end, selection evidence and exclusion log.

**Your decisions and findings**

- Method and parameters:
- Checks performed:
- Results:
- Unresolved issues / next action:


## 12. Build eight semi-monthly composites

**Work to implement**

Split each month into days 1–15 and 16–month-end with start-inclusive/end-exclusive boundaries. Median-composite valid B3/B4/B5/B8/B11 observations per pixel, then calculate the three indices from composite bands. Preserve clear-observation counts. Document this order: median indices is a different computation.

**Expected output**

Eight intervals × eight variables = 64 features per pixel, plus quality layers.

**Your decisions and findings**

- Method and parameters:
- Checks performed:
- Results:
- Unresolved issues / next action:


## 13. Apply the missing-data policy

**Work to implement**

Start with a complete-data baseline requiring all 64 features. Count usable pixels per bin, not just acquisitions over the patch. Report losses by crop, field size and region. If necessary, evaluate a predefined short-gap policy separately; do not invent observed transitions across long gaps. Fit learned imputation on training data only.

**Expected output**

Accepted feature stacks and explicit missingness/exclusion records.

**Your decisions and findings**

- Method and parameters:
- Checks performed:
- Results:
- Unresolved issues / next action:


## 14. Prepare tables and field-based partitions

**Work to implement**

Keep field_id, season_id, crop, source, region, window dates and quality as metadata. Use satellite features as model inputs and crop as target. Deduplicate fields before splitting; keep fragments, duplicate locations, pixels, windows and seasons of the same field together. Sample pixels across independent fields. Compare balanced sampling/class weights without treating correlated pixels as independent evidence.

**Expected output**

Training/validation tables and independent Benguet test fields.

**Your decisions and findings**

- Method and parameters:
- Checks performed:
- Results:
- Unresolved issues / next action:


## 15. Train and evaluate the thesis comparison

**Work to implement**

Compare external-only, limited Benguet-only, and external plus the same Benguet training fields. Train new Random Forests. Evaluate all on the same independent Benguet test fields using per-class precision/recall/F1, macro-F1, balanced accuracy and confusion matrices. Record growth-detector failures and eligible coverage. Do not tune using final test results.

**Expected output**

Reproducible model comparison, field-level sample counts and limitations.

**Your decisions and findings**

- Method and parameters:
- Checks performed:
- Results:
- Unresolved issues / next action:


## 16. Document and package outputs

**Work to implement**

Record dataset versions, licenses, preprocessing parameters, detector settings, random seeds, split definitions and export task completion. Keep raw data separate from derived outputs. Document how field-season predictions relate to the intended Benguet map period.

**Expected output**

Auditable methodology and final artifact inventory.

**Your decisions and findings**

- Method and parameters:
- Checks performed:
- Results:
- Unresolved issues / next action:


## Added experiment — Field-level cloud coverage checks

**Work to implement**

Record clear-pixel fractions and supporting counts for each field/date. Validate the acceptance threshold on development fields; inspect residual clouds and reject unreliable observations. The current 80% threshold is exploratory, not a validated universal setting. Report exclusions and coverage by crop and region.

**Expected output**

Documented comparison, quality records and limitations.

**Your decisions and findings**

- Settings:
- Checks:
- Results:
- Limitations:


## Added experiment — Feature comparison: 40 versus 64 inputs

**Work to implement**

Compare five bands × eight intervals (40 features) with five bands plus NDVI/NDRE/NDMI × eight intervals (64 features). Keep field partitions, windows, sample budgets and model selection procedures fixed. Select features only on development data; interpret correlated feature importance carefully.

**Expected output**

Documented comparison, quality records and limitations.

**Your decisions and findings**

- Settings:
- Checks:
- Results:
- Limitations:


## Added experiment — Window sensitivity analysis

**Work to implement**

Keep the green-up-aligned four-calendar-month rule as the primary design. Test a predefined shift one semi-monthly bin earlier/later using eight consecutive calendar half-month bins. Label these as sensitivity variants, not the primary four-whole-month input. Report coverage changes and do not tune window shifts on the final Benguet test set.

**Expected output**

Documented comparison, quality records and limitations.

**Your decisions and findings**

- Settings:
- Checks:
- Results:
- Limitations:


## Added experiment — Training-source comparison and per-crop reporting

**Work to implement**

Train external-only, limited Benguet-only, and external plus the same Benguet training fields. Evaluate on the same independent Benguet test fields. Report per-crop precision/recall/F1, macro-F1, balanced accuracy, confusion matrix, pixel/field counts, growth-detector failures and coverage. Keep every field together across partitions.

**Expected output**

Documented comparison, quality records and limitations.

**Your decisions and findings**

- Settings:
- Checks:
- Results:
- Limitations:


## Research and documentation

- [FranceCrops dataset and processing](https://huggingface.co/datasets/saget-antoine/francecrops)
- [L2A collection](https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S2_SR_HARMONIZED)
- [Copernicus start of season: PPI amplitude method](https://www.copernicus.eu/en/access-data/copernicus-services-catalogue/start-season-date-2017-present-raster-10-m-europe-yearly)
- [Carrot Sentinel-2 development study](https://link.springer.com/article/10.1007/s11119-023-10083-z)

Literature supports methods, not field-specific planting/harvest dates. Proposed detector parameters require validation.

## Added research references

- [Crop-mapping review](https://doi.org/10.1080/10106049.2026.2690363)
- [Feature optimization](https://www.sciencedirect.com/science/article/pii/S1569843224005284)
- [Cloud quality and cross-year yield evaluation](https://ira.agroscope.ch/de-CH/Page/Einzelpublikation/Download?einzelpublikationId=56073)

These support experimental ideas; they do not validate the exact four-month design or crop-specific onset thresholds.
